In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Load the dataset
data = pd.read_csv('AIDS_Classification_50000.csv')

# Separate features (X) and target (y)
X = data.drop(columns=['infected'])  # Assuming 'infected' is the target column
y = data['infected']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the logistic regression model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]  # Probability of the positive class

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Print evaluation metrics
print("Model Performance:")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"ROC-AUC: {roc_auc:.2f}")

# Display the confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)


Model Performance:
Accuracy: 0.69
Precision: 0.00
Recall: 0.00
F1-Score: 0.00
ROC-AUC: 0.64

Confusion Matrix:
[[6899    0]
 [3101    0]]


c:\Users\aadya\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\aadya\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\aadya\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from neural_tangents import stax
import jax.numpy as jnp

# Load dataset
data = pd.read_csv('AIDS_Classification.csv')

# Define features and target
X = data.drop(columns=['infected'])
y = data['infected']

# Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Set up K-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)  # Change n_splits to 15 for 15-fold cross-validation
accuracies = []

# Define NTK model
init_fn, apply_fn, kernel_fn = stax.serial(
    stax.Dense(128), stax.Relu(),
    stax.Dense(64), stax.Relu(),
    stax.Dense(1)
)

# K-fold cross-validation
for train_index, test_index in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Compute the NTK kernel matrix
    kernel_train = kernel_fn(X_train, X_train, "ntk")
    kernel_test = kernel_fn(X_test, X_train, "ntk")

    # Solve for the kernel regression coefficients
    coefficients = np.linalg.solve(kernel_train + 1e-3 * np.eye(len(kernel_train)), y_train)

    # Make predictions on the test set
    predictions = jnp.dot(kernel_test, coefficients)
    predictions = jnp.round(predictions)  # Convert predictions to binary

    # Evaluate the model
    accuracy = accuracy_score(y_test, predictions)
    accuracies.append(accuracy)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(accuracies)
print(f"Average Accuracy: {average_accuracy:.2f}")

Average Accuracy: 0.87
